In [1]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw
from matplotlib.path import Path
from ase import Atoms
from ase.io import write

In [44]:
file_1 = "../moire_structures/bto/bto_5-layer.vasp"
file_2 = "../moire_structures/bto/bto_5-layer.vasp"

In [45]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

In [46]:
df = pd.read_pickle('../moire_structures/bto/bto_bilayer_5l.pkl')

In [47]:
df = df.sort_values(by='norm_A1',ascending=True).head(6)

In [48]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    norm = row[1]['norm_A1'] + row[1]['norm_A2']
    replicate = int((4*norm/np.linalg.norm(lattice_vectors1['a'])))
    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 3 + (positions1_df['z'].max() - positions1_df['z'].min() )
    

    
    theta1 = row[1]['Theta']
    
    theta_rad = np.deg2rad(theta1)
    
    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])
    
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer], ignore_index=True)
    
    z_dim = (positions2_df['z'].max() - positions1_df['z'].min()) + 50
    

    A3 = np.array([0, 0, z_dim])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    polygon_points = vertex
    
    polygon_path = Path(polygon_points)
    
    selected_atoms_df = sw.select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.02)
    
    # unique_types = selected_atoms_df['type'].unique()
    
    # sort_order = {value: index for index, value in enumerate(unique_types)}
    
    # selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)
    
    # sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()
    
    # sorted_df = sorted_df.round(5)
    
    final_df = selected_atoms_df.round(5).copy()
    
    sw.write_lammps_ase(f"../moire_structures/bto/bto_tetra_{theta1}_5L.dat",A1,A2,A3, final_df)
    print(f"Done with {count} | {np.round(theta1,2)}")
    count = count + 1
    break
    
    
    


Done with 1 | 3.95
